# <u>Feature Selection</u>

### Prerequisites:

* <a href="../1.Supervised Learning/1.Regression/">Check out the notebookes on Regression</a>
* <a href="../1.Supervised Learning/2.Classification/">Check out the notebooks on Classification</a>

## Topics

* [1. Goal](#goal)
* [2. Motivation](#motivate)
    * [2.1 Size of Datasets](#size)
* [3. Feature selection vs. Feature extraction](#selection_extraction)
* [4. Types of feature selection](#types)
    * [4.1 Filter Methods](#filter)
    * [4.2 Wrapper Methods](#wrapper)
    * [4.3 Embedded Methods](#embedded)
* [5. Key Insights](#insights)
* [6. Feature Selection & Statistical Inference](#selection_inference)
* [7. How Filter Methods Work (Pipeline)](#pipeline)
* [8. Overall Takeaways](#overall)


In [2]:
import numpy as np # for arrays and random numbers

import matplotlib.pyplot as plt # for plotting

from sklearn.feature_selection import chi2 # chi square test (categorical features)

from scipy.stats import ttest_ind # welch t test

from sklearn.feature_selection import f_classif # F test

from sklearn.feature_selection import mutual_info_classif # mutual information

from sklearn.metrics import roc_auc_score # AUC / ROC (per feature)

from sklearn.feature_selection import SelectKBest # Selecting top features

from sklearn.feature_selection import SequentialFeatureSelector # Wrapper Methods (Forward / Backward Selection)

from sklearn.feature_selection import RFE # Recursive Feature Elimination (RFE)

#from mlxtend.feature_selection import SequentialFeatureSelector as SFS

#from deap import base, creator, tools # Genetic Algorithms

from sklearn.pipeline import Pipeline # pipeline

# ML models
from sklearn.neighbors import KNeighborsClassifier # k neares neighbors
from sklearn.preprocessing import PolynomialFeatures # preprocessing for Polynomial regression
from sklearn.linear_model import LinearRegression # perform OLS
from sklearn.tree import DecisionTreeClassifier # for Classification trees
from sklearn.tree import DecisionTreeRegressor # for Regression trees
from sklearn.linear_model import Ridge # Ridge regression
from sklearn.svm import SVC # (non) linear (non)separable SVM
from sklearn.linear_model import LogisticRegression # logistic regression
from sklearn.naive_bayes import GaussianNB # Naive Bayes
from sklearn.ensemble import RandomForestClassifier # Random forest classifier

print("Setup complete")

Setup complete


<a class="anchor" id="goal"></a>
# 1. Goal

**Find a well-performing, hopefully small set of features.**

Feature selection is critical for:
- reducign noise and overfitting
- improving performance/generalization
- enhancing interpretability by identifying most imformative features

#### Methods: Select features via <u>domain knowledge</u> or <u>algorithmic approaches</u>

<a class="anchor" id="motivate"></a>
# 2. Motivation

* Model is not harmed by irrelevant features since their parameters can simply be estimated as 0
* However in practice, irrelevant features can worsen generalization
- Having redundant features can cost money or time during prediction
- Many models require that sample size $n$ be greater than the number of features $(n > p)$


<a class="anchor" id="size"></a>
## 2.1 Size of Datasets

- Classical setting: If we have around $100$ features $\Rightarrow$ Feature selection might be relevant

- Medium: If we have between $100$ and $1000$ features  $\Rightarrow$ Feature selection often helpful

- High-dimensions: With more than $1000$ features $\Rightarrow$ Feature selection cruicial

<a class="anchor" id="selection_extraction"></a>
# 3. Feature selection vs. Feature extraction


<div style="display:flex; gap:20px;">

<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">


<h5 style="text-align:center;">Feature selection</h5>

- select a subset $\tilde{p}$ of $p$ features $(\tilde{p} < p)$
- keeps interpretability

<p align="center">
<img src="pics/50.png" width="600"/>
</p>


</div>


<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">

<h5 style="text-align:center;">Feature extraction</h5>

- transform $p$ features to $\tilde{p}$ with (linear / nonlinear combinations)
- information on individual features can be lost
- Example: Unsupervised PCA (multidimensional scaling,manifold learning), Supervised PCA (Partial least squares)

<p align="center">
<img src="pics/51.png" width="600"/>
</p>

</div>
</div>



<a class="anchor" id="types"></a>
# 4. Types of feature selection

<a class="anchor" id="filter"></a>
## 4.1 Filter Methods

**Evaluate relevance of features via statistics such as correlation with target variable.**

#### Mechanism of Filter Methods:
Compute relevance score for each feature $x_j$ (measure that quantifies the dependancy between features and the target variable), rank them then select top $\tilde{p}$ features. Train model on top features.


#### How to choose $\tilde{p}$:
- Arbitrarily select $\tilde{p}$ 
- Treat $\tilde{p}$  as hyperparameter and tune in a pipeline based on resampling


#### Mechanism of Filter Methods:

$$\chi^2-\text{Test}$$

- $\chi^2$-Test: Test for independence between categorical $x_j$ and categorical $y$
- Numeric features or targets can be discretized
- Hypotheses: 
$$
\begin{align*}
H_0:&p(x_j=m,y=k)=p(x_j=m)p(y=k) \hspace{2 mm} \forall m,k \text{ vs } \\

H_1:&p(x_j=m,y=k)\neq(x_j=m)p(y=k) \\

&\chi_j^2 = \sum_{m=1}^M \sum_{k=1}^K \left(\frac{e_{mk}-\tilde{e}_{mk}}{\tilde{e}_{mk}}\right) \underset{\text{approx.}}{\overset{H_0}{\sim}} \chi^2_{(M-1)(K-1)} \\
\end{align*}
$$

- $e_{mk}$ is observed relative frequency of pair $(m,k)$
- $\tilde{e}_{mk}=\frac{e_m \cdot e_k}{n}$ is expected relative frequency 
- $M$ and $K$ are number of possible values of $x_j$ and $y$
- Large $\chi_j^2$ indicates higher dependence between feature $x_j$ and target $y$ meaning higher relevance of $x_j$ 

---

$$\text{Pearson Correlation }r(x_j,y)$$
- For numeric features and numeric targets
- checks linear dependance
$$
r(x_j,y)=\frac{\overbrace{\frac{1}{n-1}\sum_{i=1}^n (x_j^{(i)}-\bar{x}_j)(y^{(i)}-\bar{y})}^{\text{Cov}(x_j,y)}}{\sqrt{\underbrace{\frac{1}{n-1}\sum_{i=1}^n (x_j^{(i)}-\bar{x}_j)^2}_{\text{Var}(x_j)}} \sqrt{\underbrace{\frac{1}{n-1}\sum_{i=1}^n (y^{(i)}-\bar{y})^2}_{\text{Var}(y)}}}
$$

- $r(x_j,y) \in [-1,1]$
- $r(x_j,y)$ closer to -1 means negative linear dependency between $x_j$ and $y$
- $r(x_j,y)$ closer to +1 means positive linear dependency between $x_j$ and $y$
- $r(x_j,y)$ closer to 0 means no linear dependency between $x_j$ and $y$
- Use absolute values $|r(x_j,y)|$ for feature ranking since direction of linear dependency is irrelevant


$$\text{Spearman Correlation }r_{SP}(x_j,y)$$
- For features and targets at least on ordinal scale
- like Preason correlation but comouted on ranks
- checks monotonicity of relationship
$$
r_{SP}(x_j,y)=
\frac{
\frac{1}{n-1}\sum_{i=1}^n
\left(R(x_j^{(i)})-\overline{R_x}\right)
\left(R(y^{(i)})-\overline{R_y}\right)
}{
\sqrt{
\frac{1}{n-1}\sum_{i=1}^n
\left(R(x_j^{(i)})-\overline{R_x}\right)^2
}
\sqrt{
\frac{1}{n-1}\sum_{i=1}^n
\left(R(y^{(i)})-\overline{R_y}\right)^2
}}
$$

---
$$\text{Welch's t-test aka two-sample-t-test for samples with unequal variances}$$

- For numeric features and binary targets $y \in \{0,1\}$
- Hypotheses: 
$$
H_0:\mu_{j_0}=\mu_{j_1} \hspace{1 mm} \text{ vs } \hspace{1 mm} H_1:\mu_{j_0}\neq\mu_{j_1} 
$$

$$
t_j = \frac{\bar{x}_{j_0}-\bar{x}_{j_1}}{\sqrt{\frac{S_{x_{j_0}}^2}{n_0} + \frac{S_{x_{j_1}}^2}{n_1}}}
$$

- $\bar{x}_{j_0}$ and $\bar{x}_{j_1}$ are sample mean for $y=0$ and $y=1$
- $S_{x_{j_0}}^2$ are sample variance for $y=0$ and $y=1$
- $n_0$ and $n_1$ are sample sizes for $y=0$ and $y=1$
- Higher value for $t_j$ indicates higher relevance of $x_j$
---
$$\text{AUC/ROC}$$

- For binary classification with $y \in \{0,1\}$ and numeric features
- Compute AUC score for every feature (with tresholds) and its ability to separate classes
- Rank features and features with higher AUC score mean higher relevance

---
$$\text{F-test}$$

- For multiclass classification ($g \geq 2$) and numeric features
- Measures how much the expected values per feature $x_j$ within the classes of the target variable differ from each other
- Hypotheses: 
$$
H_0:\mu_{j_0}=\mu_{j_1}=\ldots=\mu_{j_g}= \hspace{1 mm} \text{ vs } \hspace{1 mm} H_1:\exists k,l: \mu_{j_k}\neq\mu_{j_l}
$$

$$
\begin{align*}
F &= \frac{\text{between-group variability}}{\text{within-group variability}} \\
&= \frac{\sum_{k=1}^g n_k (\bar{x}_{j_k}-\bar{x}_j)^2 / (g-1)}{\sum_{k=1}^g \sum_{i=1}^{n_k}(x_{j_k}^{(i)}-\bar{x}_{j_k})^2/(n-g)}
\end{align*}
$$

- $\bar{x_{j_k}}$ is sample mean of feature $x_j$ where $y=k$
- $\bar{x}_j$ is overall sample mean of feature $x_j$
- High F-score means higher relevance of the feature $x_j$

---
$$\text{Mutual information (MI)}$$
- Each feature $x_j$ is rated according to $\underbrace{I(X,Y)=\mathbb{E}_{p(x,y)}\left[\log\left(\frac{p(X,Y)}{p(X)p(Y)}\right)\right]}_{\text{Information gain}}$

- Measures amount of dependence by measuring difference between joint distribution $p(X,Y)$ from their independence $p(X)p(Y)$

- MI = 0 if $X$ is independent from $Y$ 
- MI is large when $X$ and $Y$ are dependent or vice versa
- MI is for both numeric and categorical features and targets
- Estimate $p(X,Y)$, $p(X)$ and $p(Y)$ as observed frequencies (for discrete features)
- Estimate $p(X,Y)$, $p(X)$ and $p(Y)$ through binning or kernel density estimation (for continuous features)



<div style="display:flex; gap:20px;">

<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">


##### Advantages
- scales well with increasing number of features $p$
- generally interpretable

</div>


<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">

##### Disadvantages

- Filters can be misleading
- Redundant features may still add value when combined with others
- Some features useless alone may be useful together 


</div>
</div>


<a class="anchor" id="wrapper"></a>
## 4.2 Wrapper Methods

**Use models to evaluate subset of features.**

- Use a leaners performance as evaluation metric for a subset of features
- Given $p$ features find subset $S \subset \{1,\ldots,p\}$ optimizing some objective $\psi : \Omega \rightarrow \mathbb{R}:S^* \in \arg \min_{S \in \Omega} \left\{\psi(S)\right\}$
    - $\Omega$ = Search space of all feature subsets $S \subset \{1,\ldots,p\}$
        - usually encoded by bit vectors, i.e. $\Omega=\{0,1\}^p$ (1 = feature selected)
    - Objective $\psi$ can be different functions such cross validated performance of a learner

**Problem:** Discrete combinatorial optimization problem over search space of size = $2^p$, i.e. grows exponentially in $p$.

**Solution:** Avoid searching entire space by employing efficient search strategies.




## Greedy forward search (GFS)
**Start empty and add best feature iteratively and terminate if performance does not improve further or max number of fratures is used.**

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/52.png" width="550"/>
  <img src="pics/53.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/54.png" width="550"/>
  <img src="pics/55.png" width="550"/>
</div>



## Greedy backward search (GBS)
**Start full and remove worst feature iteratively.**

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/56.png" width="550"/>
  <img src="pics/57.png" width="550"/>
</div>


## Extensions Algorithms (EAs)
**Wrapper feature selection methods can be accelerated and improved by adding/removing multiple features simultaneously, alternating forward and backward search, randomly sampling feature subsets, and focusing the search on promising regions of the feature space.**

<p align="center">
<img src="pics/58.png" width="500"/>
</p>





<div style="display:flex; gap:20px;">

<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">


##### Advantages
- Can be combined with any learner
- Any performance measure can be used
- Optimizes desired criterion directly

</div>


<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">

##### Disadvantages

- Does not scale well with growing numbers of features $p$
- Computationlly expensive
- Requires nested resampling


</div>
</div>

In [ ]:
# Greed Forward Search

In [ ]:
# Greed Backward Search

In [ ]:
# EAs

<a class="anchor" id="embedded"></a>
## 4.3 Embedded Methods

**Integrate Feature Selection directly into specific model.**

- Example: Lasso with L1 penalty performs automatic feature selction
$$
\mathcal{R}_\text{reg}(\theta)=\mathcal{R}_\text{emp}(\theta) + \lambda \lVert \theta \rVert_1 = \sum_{i=1}^n (y^{(i)}-\theta^\top x^{(i)})^2 + \lambda \sum_{i=1}^p |\theta_j|
$$


<a class="anchor" id="insights"></a>
# 5. Key Insights

<a class="anchor" id="pipeline"></a>
# 6. How Filter Methods Work (Pipeline)

1. Compute score for each feature
2. Rank features
3. Select top $\tilde{p}$ features
4. Train model

Choosing number of features:

- Based on domain knowledge
- Visual inspection
- Hyperparameter tuning

<a class="anchor" id="overall"></a>
# 7. Overall Takeaways

- Too many features can hurt performance
- Feature selection is crucial in high-dimensional data
- No single best method:
    - Filters $\rightarrow$ fast but simple
    - Wrappers $\rightarrow$ accurate but expensive
    - Embedded $\rightarrow$ efficient compromise
- Feature interactions are important $\rightarrow$ univariate methods can miss them
- Feature selection affects both prediction and statistical inference